# 5.1 dbt with Python Models> dbt is the standard for analytics engineering. Python models extend it beyond SQL.---

## dbt Overviewdbt (data build tool) lets you transform data in your warehouse using SQL and Python models.```my_dbt_project/├── models/│   ├── staging/│   │   └── stg_patients.sql       # SQL model│   ├── marts/│   │   └── readmission_risk.py    # Python model│   └── schema.yml                 # tests & docs├── dbt_project.yml└── profiles.yml```

In [ ]:
# Example: dbt Python model (runs as PySpark/Snowpark in warehouse)# File: models/marts/readmission_features.pydbt_model = '''def model(dbt, session):    """Build readmission risk features from staged data."""    # Reference another dbt model (like {{ ref('stg_patients') }} in SQL)    patients = dbt.ref("stg_patients")    encounters = dbt.ref("stg_encounters")        # Join and compute features using DataFrame API    features = (        patients        .join(encounters, on="patient_id")        .groupBy("patient_id", "age", "diagnosis")        .agg(            F.count("encounter_id").alias("num_encounters"),            F.avg("cost").alias("avg_cost"),            F.max("admission_date").alias("last_visit"),        )    )        return features'''print(dbt_model)

In [ ]:
# schema.yml — data tests and documentationschema_yml = '''version: 2models:  - name: readmission_features    description: "Patient features for readmission risk prediction"    columns:      - name: patient_id        description: "Unique patient identifier"        tests:          - unique          - not_null      - name: age        tests:          - not_null      - name: num_encounters        tests:          - not_null          - dbt_utils.accepted_range:              min_value: 1'''print(schema_yml)